In [ ]:
# setup
! uv pip install -U fire hydra-core wandb jax[tpu] flax -q
! git clone --depth=1 https://github.com/martin-marek/picodo.git
! python /content/picodo/download_fineweb.py 'fineweb' 26 # 2.6B tokens

In [ ]:
# imports
%cd /content/picodo
import os
from hydra import compose, initialize_config_dir
from configs import resolver_setup
from train import train_and_evaluate

In [ ]:
# training (124M)
with initialize_config_dir(f'{os.getcwd()}/configs', version_base=None):
    c = compose(config_name='base', overrides=['+model=gpt2s', '+dataset=fw_gpt2'])
c.model.activ_dtype = 'bfloat16' # reduced precision increases arithmetic intensity
c.model.H = 256 # TPU v6e has 256x256 MXU
c.opt.batch_size = 8 # smallest batch size maximizing MFU
c.opt.peak_lr = 0.002
c.opt.b2 = 0.999 # small batch size requires high b2 (arxiv.org/abs/2507.07101)
c.opt.weight_decay = 0.02 # small batch size requires small wd
c.wandb_mode = 'disabled' # ['disabled', 'offline', 'online']
train_and_evaluate(c)